In [6]:
import arcpy
import collections
import os

def main():
    # ================= 參數設定區 (請依需求修改) =================
    input_layer = "OL_P1_2209_cor_SimplifyPolyg"  # 你的原始 Polygon 圖層名稱
    out_point_fc = "Recruit_Centroids_2204"    # 輸出的中心點圖層名稱 (現在會永久保留)
    out_square_fc = "Recruit_Squares_2204"     # 輸出的正方形圖層名稱
    
    # 欄位名稱對應 (原始欄位名稱)
    f_recruit = "Recruit"
    f_id = "ID"
    f_form = "Form"
    f_area = "Shape_Area"
    
    # 正方形邊長列表 (單位: 公分)
    square_sizes_cm = [2, 4, 8, 16, 32, 64, 128]
    
    # 設定環境
    arcpy.env.overwriteOutput = True
    
    # 取得輸入圖層的空間參考
    desc_input = arcpy.Describe(input_layer)
    sr = desc_input.spatialReference
    
    # 檢查座標系統單位 (為了正確轉換公分)
    unit_factor = 1.0
    if sr.linearUnitName.lower() in ['meter', 'metre']:
        unit_factor = 0.01  # 1 cm = 0.01 m
    elif sr.linearUnitName.lower() in ['foot', 'feet']:
        unit_factor = 0.0328084 # 1 cm = 0.0328 ft
    else:
        arcpy.AddWarning("⚠️ 警告：座標系統非公尺或英尺，請確認單位換算。目前假設單位為公尺。")
        unit_factor = 0.01

    # ================= 1. 處理 Domain (代碼轉文字) =================
    msg = "正在讀取屬性域 (Domain) 資訊..."
    arcpy.AddMessage(msg)
    print(msg)
    
    # 獲取 Form 欄位使用的 Domain 名稱
    fields = arcpy.ListFields(input_layer)
    form_domain_name = None
    for field in fields:
        if field.name == f_form:
            form_domain_name = field.domain
            break
            
    # 建立 代碼->文字 的對照表
    domain_lookup = {}
    if form_domain_name:
        # 從輸入圖層的路徑尋找資料庫以獲取 Domain
        workspace_path = desc_input.path
        domains = arcpy.da.ListDomains(workspace_path)
        for domain in domains:
            if domain.name == form_domain_name:
                if domain.domainType == 'CodedValue':
                    domain_lookup = domain.codedValues
                break
    else:
        warn_msg = f"注意：欄位 {f_form} 沒有設定 Domain，將直接使用原始值。"
        arcpy.AddWarning(warn_msg)
        print(warn_msg)

    # ================= 2. 生成中心點並轉移屬性 (永久存儲) =================
    msg = f"正在篩選 Polygon 並生成點圖層: {out_point_fc}..."
    arcpy.AddMessage(msg)
    print(msg)
    
    # 在當前工作空間建立輸出點圖層 (不再使用 memory)
    arcpy.management.CreateFeatureclass(arcpy.env.workspace, out_point_fc, "POINT", spatial_reference=sr)
    
    # 新增欄位到點圖層
    arcpy.management.AddField(out_point_fc, f_id, "TEXT") 
    arcpy.management.AddField(out_point_fc, "Form_Desc", "TEXT")
    arcpy.management.AddField(out_point_fc, "Orig_Area", "DOUBLE")

    # 準備統計數據
    stats_counter = collections.Counter()
    total_points = 0
    points_data = [] # 暫存點資料以便後續生成正方形

    # 讀取原始資料
    fields_to_read = [f_recruit, "SHAPE@", f_id, f_form, f_area]
    
    # 插入點資料
    fields_to_insert = ["SHAPE@", f_id, "Form_Desc", "Orig_Area"]
    with arcpy.da.InsertCursor(out_point_fc, fields_to_insert) as insert_cursor:
        with arcpy.da.SearchCursor(input_layer, fields_to_read) as search_cursor:
            for row in search_cursor:
                recruit_val = row[0]
                
                # 篩選條件：Recruit 為 "Y"
                if recruit_val and str(recruit_val).strip().upper() == "Y":
                    geom = row[1]
                    oid = row[2]
                    raw_form = row[3]
                    area_val = row[4]
                    
                    centroid = geom.centroid # 取得幾何中心
                    form_text = domain_lookup.get(raw_form, str(raw_form)) # Domain 轉換
                    
                    # 寫入點圖層
                    insert_cursor.insertRow([centroid, oid, form_text, area_val])
                    
                    # 記錄到記憶體中供後續正方形生成使用
                    points_data.append({
                        "geom": centroid, 
                        "id": oid, 
                        "form": form_text, 
                        "area": area_val
                    })
                    stats_counter[form_text] += 1
                    total_points += 1

    msg = f"已生成 {total_points} 個中心點於 {out_point_fc}。"
    arcpy.AddMessage(msg)
    print(msg)

    if total_points == 0:
        print("沒有符合條件 (Recruit='Y') 的圖徵，程式結束。")
        return

    # ================= 3. 生成多層級正方形 =================
    msg = f"正在生成正方形圖層: {out_square_fc}..."
    arcpy.AddMessage(msg)
    print(msg)
    
    # 在當前工作空間建立輸出正方形圖層
    arcpy.management.CreateFeatureclass(arcpy.env.workspace, out_square_fc, "POLYGON", spatial_reference=sr)
    
    # 新增欄位
    arcpy.management.AddField(out_square_fc, f_id, "TEXT")
    arcpy.management.AddField(out_square_fc, "Form_Desc", "TEXT")
    arcpy.management.AddField(out_square_fc, "Orig_Area", "DOUBLE")
    arcpy.management.AddField(out_square_fc, "Size_Level_cm", "SHORT")

    # 寫入正方形資料
    sq_fields = ["SHAPE@", f_id, "Form_Desc", "Orig_Area", "Size_Level_cm"]
    with arcpy.da.InsertCursor(out_square_fc, sq_fields) as cursor:
        for pt_data in points_data:
            center_pt = pt_data["geom"]
            cx, cy = center_pt.X, center_pt.Y
            
            for size_cm in square_sizes_cm:
                # 計算正方形的一半邊長
                half_side = (size_cm * unit_factor) / 2.0
                
                # 建構四個角點
                p1 = arcpy.Point(cx - half_side, cy - half_side)
                p2 = arcpy.Point(cx - half_side, cy + half_side)
                p3 = arcpy.Point(cx + half_side, cy + half_side)
                p4 = arcpy.Point(cx + half_side, cy - half_side)
                
                square_geom = arcpy.Polygon(arcpy.Array([p1, p2, p3, p4, p1]), sr)
                
                cursor.insertRow([
                    square_geom, 
                    pt_data["id"], 
                    pt_data["form"], 
                    pt_data["area"], 
                    size_cm
                ])

    # ================= 4. 最終回報 =================
    report = ["\n" + "="*30]
    report.append(f"處理完成報告：")
    report.append(f"1. 點圖層已儲存於: {out_point_fc}")
    report.append(f"2. 正方形圖層已儲存於: {out_square_fc}")
    report.append(f"共使用了 {total_points} 個點生成正方形序列。")
    report.append("-" * 40)
    report.append(f"{'Form 類別':<15} | {'數量':<8} | {'比例':<8}")
    report.append("-" * 40)
    
    for form_type, count in stats_counter.most_common():
        percentage = (count / total_points) * 100
        report.append(f"{str(form_type):<15} | {count:<8} | {percentage:.1f}%")
        
    report.append("="*30 + "\n")
    
    for line in report:
        arcpy.AddMessage(line)
        print(line)

if __name__ == '__main__':
    main()

正在讀取屬性域 (Domain) 資訊...
正在篩選 Polygon 並生成點圖層: Recruit_Centroids_2204...
已生成 142 個中心點於 Recruit_Centroids_2204。
正在生成正方形圖層: Recruit_Squares_2204...

處理完成報告：
1. 點圖層已儲存於: Recruit_Centroids_2204
2. 正方形圖層已儲存於: Recruit_Squares_2204
共使用了 142 個點生成正方形序列。
----------------------------------------
Form 類別         | 數量       | 比例      
----------------------------------------
fo              | 55       | 38.7%
en              | 25       | 17.6%
cb              | 23       | 16.2%
su              | 22       | 15.5%
br              | 13       | 9.2%
di              | 3        | 2.1%
ma              | 1        | 0.7%



In [1]:
import arcpy
import os

# --- 1. 設定參數 ---
rect_fc = "Recruit_Squares_2204"
overlap_fc_t1 = "OL_P1_2204_coral_Simplify"  # 第 1 個時段 (T1)
overlap_fc_t2 = "OL_P1_2209_cor_SimplifyPolyg"  # 第 2 個時段 (T2)
boundary_fc = "OL_P1_plot_6m"
form_field = "Form"

# 輸出 CSV 設定
output_folder = r"D:\ArcGIS\Morpho_Spatial_Demography\output"  # !!! 請修改為您的資料夾路徑 !!!
csv_name = "Recruit_cover_change.csv"

# ID 與欄位參數
id_part1 = "ID"
id_part2 = "Size_Level_cm"
custom_uid_field = "Recruit_UID" 
orig_area_field = "Orig_Area"  # 原始矩形生成時建立的面積欄位

# 新增計算欄位名稱
t2_orig_gap_field = "T2_Orig_Gap"
net_growth_field = "Net_Growth"

area_unit = "SQUARE_METERS"
out_boundary_field = "Exceeds_Boundary"

arcpy.env.overwriteOutput = True

# --- 2. 建立合併的唯一 ID (Recruit_UID) ---
if not arcpy.ListFields(rect_fc, custom_uid_field):
    arcpy.management.AddField(rect_fc, custom_uid_field, "TEXT", field_length=100)

print("正在更新 Recruit_UID...")
expression = f"str(!{id_part1}!) + '_' + str(!{id_part2}!)"
arcpy.management.CalculateField(rect_fc, custom_uid_field, expression, "PYTHON3")

# --- 3. 自動讀取 Domain ---
def get_domain_map(fc, field_name):
    desc = arcpy.Describe(fc)
    d_name = next((f.domain for f in desc.fields if f.name == field_name), "")
    if d_name:
        workspace = desc.path
        domains = arcpy.da.ListDomains(workspace)
        for d in domains:
            if d.name == d_name:
                return {k: arcpy.ValidateFieldName(v) for k, v in d.codedValues.items()}
    return {}

domain_map = get_domain_map(overlap_fc_t1, form_field)
all_forms = list(domain_map.values())
print(f"解析到 {len(all_forms)} 種珊瑚類別。")

# --- 4. 定義運算函數 ---
def run_intersection(in_fc, suffix):
    print(f"正在運算 {suffix} 的重疊面積...")
    temp_table = f"memory/table_{suffix}"
    arcpy.analysis.TabulateIntersection(
        in_zone_features=rect_fc,
        zone_fields=custom_uid_field,
        in_class_features=in_fc,
        out_table=temp_table,
        class_fields=form_field,
        out_units=area_unit
    )
    
    period_results = {}
    total_col = f"Total_Area_{suffix}"
    
    with arcpy.da.SearchCursor(temp_table, [custom_uid_field, form_field, "AREA"]) as sc:
        for row in sc:
            uid, f_code, area = row
            f_alias = f"{domain_map.get(f_code, str(f_code))}_{suffix}"
            if uid not in period_results:
                period_results[uid] = {total_col: 0}
            period_results[uid][f_alias] = area
            period_results[uid][total_col] += area
    return period_results

# 執行 T1 與 T2 運算
results_t1 = run_intersection(overlap_fc_t1, "T1")
results_t2 = run_intersection(overlap_fc_t2, "T2")

# --- 5. 準備所有欄位 ---
t1_fields = [f"{f}_T1" for f in all_forms] + ["Total_Area_T1"]
t2_fields = [f"{f}_T2" for f in all_forms] + ["Total_Area_T2"]
calc_fields = [t2_orig_gap_field, net_growth_field]

for f in t1_fields + t2_fields + calc_fields:
    if not arcpy.ListFields(rect_fc, f):
        arcpy.management.AddField(rect_fc, f, "DOUBLE")

if not arcpy.ListFields(rect_fc, out_boundary_field):
    arcpy.management.AddField(rect_fc, out_boundary_field, "SHORT")

# --- 6. 判斷範圍 ---
print("檢查範圍邊界...")
rect_lyr = arcpy.management.MakeFeatureLayer(rect_fc, "rect_lyr")
arcpy.management.SelectLayerByLocation(rect_lyr, "COMPLETELY_WITHIN", boundary_fc)
within_uids = {row[0] for row in arcpy.da.SearchCursor(rect_lyr, [custom_uid_field])}

# --- 7. 更新數據與數學計算 ---
print("正在計算並寫入數據...")
# 這裡加入 orig_area_field 到讀取清單中
update_fields = [custom_uid_field, out_boundary_field, orig_area_field] + t1_fields + t2_fields + calc_fields
f_idx = {name: i for i, name in enumerate(update_fields)}

with arcpy.da.UpdateCursor(rect_fc, update_fields) as cursor:
    for row in cursor:
        uid = row[f_idx[custom_uid_field]]
        orig_area = row[f_idx[orig_area_field]] if row[f_idx[orig_area_field]] else 0
        
        # A. 範圍標記
        row[f_idx[out_boundary_field]] = 0 if uid in within_uids else 1
        
        # B. 寫入 T1 數據
        area_t1 = 0
        if uid in results_t1:
            area_t1 = results_t1[uid].get("Total_Area_T1", 0)
            for f in t1_fields:
                row[f_idx[f]] = results_t1[uid].get(f, 0)
        else:
            for f in t1_fields: row[f_idx[f]] = 0
            
        # C. 寫入 T2 數據
        area_t2 = 0
        if uid in results_t2:
            area_t2 = results_t2[uid].get("Total_Area_T2", 0)
            for f in t2_fields:
                row[f_idx[f]] = results_t2[uid].get(f, 0)
        else:
            for f in t2_fields: row[f_idx[f]] = 0
        
        # D. 新增的數學計算
        # 1. Total_Area_T2 - Orig_Area
        row[f_idx[t2_orig_gap_field]] = area_t2 - orig_area
        
        # 2. Total_Area_T2 - Total_Area_T1 - Orig_Area
        row[f_idx[net_growth_field]] = area_t2 - area_t1 - orig_area
            
        cursor.updateRow(row)

# --- 8. 輸出 CSV ---
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

csv_output_full = os.path.join(output_folder, csv_name)
print(f"正在匯出 CSV 到: {csv_output_full}")

# 使用 TableToTable 匯出為 CSV
arcpy.conversion.TableToTable(rect_fc, output_folder, csv_name)

arcpy.management.Delete("rect_lyr")
print("🎉 任務成功完成！CSV 已輸出。")

正在更新 Recruit_UID...
解析到 12 種珊瑚類別。
正在運算 T1 的重疊面積...
正在運算 T2 的重疊面積...
檢查範圍邊界...
正在計算並寫入數據...
正在匯出 CSV 到: D:\ArcGIS\Morpho_Spatial_Demography\output\Recruit_cover_change.csv
🎉 任務成功完成！CSV 已輸出。


In [1]:
import arcpy

# 1. 設定參數
point_feature = "Recruit_Centroids_2204"  
polygon_layer = "OL_P1_2209_cor_SimplifyPolyg"                
t1_layer = "OL_P1_2204_coral_Simplify"  # <--- 新增：前一次調查的珊瑚圖層 (T1)

source_area_field = "Shape_Area"
source_form_field = "Form"
target_area_field = "Poly_Area"        
target_form_field = "Poly_Form_Text"  
target_t1_field = "In_T1_Layer"        # <--- 新增：存入 Y/N 的欄位名稱

# 取得點圖徵的 OID 欄位
oid_fieldname = arcpy.Describe(point_feature).OIDFieldName

# --- 2. 建立 Domain 對照表 ---
desc = arcpy.Describe(polygon_layer)
workspace = desc.path
if arcpy.Describe(workspace).dataType == "FeatureDataset":
    workspace = arcpy.Describe(workspace).path

domain_map = {}
field_objs = arcpy.ListFields(polygon_layer, source_form_field)
if field_objs and field_objs[0].domain:
    target_domain_name = field_objs[0].domain
    domains = arcpy.da.ListDomains(workspace)
    for domain in domains:
        if domain.name == target_domain_name:
            domain_map = domain.codedValues
            break

# --- 3. 判斷點是否落在 T1 圖層 ---
print(f"正在比對點圖徵是否落在 {t1_layer}...")
t1_set = set()
point_lyr = "temp_point_lyr"
arcpy.management.MakeFeatureLayer(point_feature, point_lyr)
# 選取與 T1 相交的點
arcpy.management.SelectLayerByLocation(point_lyr, "INTERSECT", t1_layer)

# 將選中的 OID 存入 set
with arcpy.da.SearchCursor(point_lyr, [oid_fieldname]) as sc:
    for row in sc:
        t1_set.add(row[0])

# --- 4. 設定 FieldMappings (空間連接 polygon_layer) ---
print("正在設定欄位對照...")
field_mappings = arcpy.FieldMappings()
field_mappings.addTable(point_feature)

fm_area = arcpy.FieldMap()
fm_area.addInputField(polygon_layer, source_area_field)
new_field = fm_area.outputField
new_field.name = "JOIN_AREA"
fm_area.outputField = new_field
field_mappings.addFieldMap(fm_area)

fm_form = arcpy.FieldMap()
fm_form.addInputField(polygon_layer, source_form_field)
new_field_f = fm_form.outputField
new_field_f.name = "JOIN_FORM"
fm_form.outputField = new_field_f
field_mappings.addFieldMap(fm_form)

# --- 5. 執行批次空間連接 (對應第一個 Polygon) ---
temp_join_result = r"memory\spatial_join_temp"
print("正在執行空間連接...")
arcpy.analysis.SpatialJoin(
    target_features=point_feature,
    join_features=polygon_layer,
    out_feature_class=temp_join_result,
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_COMMON",
    field_mapping=field_mappings,
    match_option="INTERSECT"
)

data_lookup = {}
with arcpy.da.SearchCursor(temp_join_result, ["TARGET_FID", "JOIN_AREA", "JOIN_FORM"]) as s_cursor:
    for row in s_cursor:
        pt_oid, area_val, form_code = row
        form_text = domain_map.get(form_code, str(form_code)) if form_code is not None else "NAN"
        data_lookup[pt_oid] = (area_val, form_text)

# --- 6. 自動檢查並建立欄位 ---
existing_fields = [f.name for f in arcpy.ListFields(point_feature)]
if target_area_field not in existing_fields:
    arcpy.management.AddField(point_feature, target_area_field, "DOUBLE")
if target_form_field not in existing_fields:
    arcpy.management.AddField(point_feature, target_form_field, "TEXT", field_length=100)
if target_t1_field not in existing_fields:
    arcpy.management.AddField(point_feature, target_t1_field, "TEXT", field_length=10)

# --- 7. 更新原始點圖徵並統計 ---
print(f"正在更新點圖徵並計算統計量...")
count_y = 0
count_n = 0

# 加入 target_t1_field 到 cursor
update_fields = [oid_fieldname, target_area_field, target_form_field, target_t1_field]
with arcpy.da.UpdateCursor(point_feature, update_fields) as u_cursor:
    for row in u_cursor:
        current_oid = row[0]
        
        # 更新 Polygon 1 資訊 (Area & Form)
        if current_oid in data_lookup:
            row[1], row[2] = data_lookup[current_oid]
        
        # 更新 T1 資訊 (Y/N)
        if current_oid in t1_set:
            row[3] = "Y"
            count_y += 1
        else:
            row[3] = "N"
            count_n += 1
            
        u_cursor.updateRow(row)

# 清理暫存
arcpy.management.Delete(temp_join_result)
arcpy.management.Delete(point_lyr)

# --- 8. 報表輸出 ---
print("-" * 30)
print(f"任務完成！統計結果如下：")
print(f"落在 T1 圖層內的點數量 (Y): {count_y}")
print(f"不在 T1 圖層內的點數量 (N): {count_n}")
print(f"總處理點數: {count_y + count_n}")
print("-" * 30)

正在比對點圖徵是否落在 OL_P1_2204_coral_Simplify...
正在設定欄位對照...
正在執行空間連接...
正在更新點圖徵並計算統計量...
------------------------------
任務完成！統計結果如下：
落在 T1 圖層內的點數量 (Y): 24
不在 T1 圖層內的點數量 (N): 118
總處理點數: 142
------------------------------
